In [ ]:
%pip install polars huggingface_hub risenlab-agentlogs

# Load the dataset

This notebook reads parquet from a local directory. Run **one** of the next two cells.

The sample under `data/dataset-sample/` is included in this repository and is enough to follow the examples.

In [ ]:
from pathlib import Path

from agentlogs.schema import assert_dataset_version

dataset_path = Path(".") / ".." / ".." / "data" / "dataset-sample"
assert_dataset_version(dataset_path)

For the full tables, download a Hugging Face snapshot into `data/dataset/` (files already present are skipped).

In [ ]:
from pathlib import Path

from huggingface_hub import snapshot_download
from agentlogs.schema import assert_dataset_version

dataset_path = Path(".") / ".." / ".." / "data" / "dataset"
snapshot_download(
    repo_id="risenlab/agentlogs",
    repo_type="dataset",
    revision="v0.2",
    local_dir=dataset_path,
)
assert_dataset_version(dataset_path)

# Setup

In [ ]:
import polars as pl

log_parts = sorted((dataset_path / "agent_session_logs").glob("*.parquet"))
table_path = {
    "repositories": dataset_path / "repositories" / "*.parquet",
    "agent_tasks": dataset_path / "agent_tasks" / "*.parquet",
    "agent_sessions": dataset_path / "agent_sessions" / "*.parquet",
    "agent_session_logs": dataset_path / "agent_session_logs" / "*.parquet",
    "agent_session_logs_shard": log_parts[0],
    "users": dataset_path / "users" / "*.parquet",
}

# Repositories

In [ ]:
repos = pl.scan_parquet(str(table_path["repositories"]))
repo_stats = repos.select(
    pl.len().alias("n"),
    pl.col("agent_tasks").list.len().gt(0).sum().alias("n_with_tasks"),
).collect()
n_repositories = repo_stats["n"][0]
n_repositories_with_tasks = repo_stats["n_with_tasks"][0]

print(f"# repositories: {n_repositories}")
print(f"# repositories with agent tasks: {n_repositories_with_tasks}")
print(f"% repositories with agent tasks: {100.0 * n_repositories_with_tasks / n_repositories:.2f}%")

# Agent tasks

In [ ]:
tasks = pl.scan_parquet(str(table_path["agent_tasks"]))
task_stats = tasks.select(
    pl.len().alias("n"),
    pl.col("found").sum().alias("n_found"),
    pl.col("sessions").list.len().gt(0).sum().alias("n_with_sessions"),
).collect()
n_tasks = task_stats["n"][0]
n_tasks_found = task_stats["n_found"][0]
n_tasks_with_sessions = task_stats["n_with_sessions"][0]

print(f"# tasks: {n_tasks}")
print(f"# tasks found: {n_tasks_found}")
print(f"% tasks found: {100.0 * n_tasks_found / n_tasks:.2f}%")
print(f"# tasks with agent sessions: {n_tasks_with_sessions}")
print(f"% tasks with agent sessions: {100.0 * n_tasks_with_sessions / n_tasks_found:.2f}%")

# Agent sessions

In [ ]:
sessions = pl.scan_parquet(str(table_path["agent_sessions"]))
session_stats = sessions.select(
    pl.len().alias("n"),
    pl.col("log_found").sum().alias("n_logs_found"),
).collect()
n_sessions = session_stats["n"][0]
n_logs_found = session_stats["n_logs_found"][0]
n_sessions_with_nonempty_logs = (
    pl.scan_parquet(str(table_path["agent_session_logs"]))
    .select(pl.col("session").struct.field("id").n_unique())
    .collect()
    .item()
)

print(f"# sessions: {n_sessions}")
print(f"# sessions with logs found: {n_logs_found}")
print(f"% sessions with logs found: {100.0 * n_logs_found / n_sessions:.2f}%")
print(f"# sessions with non-empty logs: {n_sessions_with_nonempty_logs}")
print(f"% sessions with non-empty logs: {100.0 * n_sessions_with_nonempty_logs / n_logs_found:.2f}%")

# Agent session logs

In [ ]:
logs = pl.scan_parquet(str(table_path["agent_session_logs"]))
log_stats = logs.select(
    pl.len().alias("n"),
    pl.col("parsed").sum().alias("n_parsed"),
).collect()
n_log_entries = log_stats["n"][0]
n_log_entries_parsed = log_stats["n_parsed"][0]

print(f"# log entries: {n_log_entries}")
print(f"# log entries parsed: {n_log_entries_parsed}")
print(f"% log entries parsed: {100.0 * n_log_entries_parsed / n_log_entries:.2f}%")

## Identify different kinds of tool calls

Uses the first `agent_session_logs` parquet shard only (see Setup).

In [ ]:
tool_calls = (
    pl.scan_parquet(str(table_path["agent_session_logs_shard"]))
    .filter(pl.col("parsed"), pl.col("data").is_not_null())
    .select(pl.col("data").struct.field("choices").alias("choices"))
    .explode("choices", empty_as_null=True)
    .select(
        pl.col("choices").struct.field("delta").struct.field("tool_calls").alias("delta_tool_calls"),
        pl.col("choices").struct.field("message").struct.field("tool_calls").alias("message_tool_calls"),
    )
)
delta_calls = (
    tool_calls.select(pl.col("delta_tool_calls").alias("tool_calls"))
    .explode("tool_calls", empty_as_null=True)
    .filter(pl.col("tool_calls").is_not_null())
    .select(
        pl.lit("delta").alias("source"),
        pl.coalesce(
            pl.col("tool_calls").struct.field("function_name"),
            pl.col("tool_calls").struct.field("custom_name"),
        ).alias("function_name"),
        pl.col("tool_calls").struct.field("type").alias("type"),
    )
)
message_calls = (
    tool_calls.select(pl.col("message_tool_calls").alias("tool_calls"))
    .explode("tool_calls", empty_as_null=True)
    .filter(pl.col("tool_calls").is_not_null())
    .select(
        pl.lit("message").alias("source"),
        pl.col("tool_calls").struct.field("function_name").alias("function_name"),
        pl.col("tool_calls").struct.field("type").alias("type"),
    )
)
df = (
    pl.concat([delta_calls, message_calls])
    .filter(pl.col("function_name").is_not_null())
    .group_by("function_name", "type", "source")
    .agg(pl.len().alias("n"))
    .with_columns(
        (100.0 * pl.col("n") / pl.col("n").sum().over(pl.lit(1))).round(4).alias("pct")
    )
    .sort("n", descending=True)
    .collect()
)
df

# Users

In [ ]:
users = pl.scan_parquet(str(table_path["users"]))
user_stats = users.select(
    pl.len().alias("n"),
    pl.col("found").sum().alias("n_found"),
).collect()
n_users = user_stats["n"][0]
n_users_found = user_stats["n_found"][0]

print(f"# users: {n_users}")
print(f"# users found: {n_users_found}")
print(f"% users found: {100.0 * n_users_found / n_users:.2f}%")